---
title: "Where LangGraph Belongs"
draft: true
categories: [agents, workflows, langgraph]
---


A model API, typed tools, and a small Python loop are enough for many tasks. A graph earns its place only when state transitions need names, checkpoints, parallel joins, stage-specific recovery, or review. This chapter makes that boundary executable before introducing the framework.

## The task-boundary test

Represent a task by six operational signals. The threshold below is deliberately transparent: two signals justify evaluating a graph, but the final choice still depends on whether the added control surface pays for itself.


In [1]:
from dataclasses import dataclass
from IPython.display import Markdown, display

@dataclass(frozen=True)
class TaskProfile:
    name: str
    typed_handoff: bool = False
    pause_resume: bool = False
    parallel_join: bool = False
    stage_recovery: bool = False
    replay: bool = False
    completion_contract: bool = False

    @property
    def signals(self) -> int:
        return sum((self.typed_handoff, self.pause_resume, self.parallel_join,
                    self.stage_recovery, self.replay, self.completion_contract))

    @property
    def choice(self) -> str:
        return "evaluate a graph" if self.signals >= 2 else "keep plain Python"

profiles = [
    TaskProfile("FAQ"),
    TaskProfile("code refactor", typed_handoff=True, stage_recovery=True),
    TaskProfile("research brief", True, True, True, True, True, True),
    TaskProfile("incident postmortem", True, True, True, True, True, True),
]
rows = "\n".join(f"| {p.name} | {p.signals} | {p.choice} |" for p in profiles)
display(Markdown("| Task | Signals | Decision |\n|---|---:|---|\n" + rows))


| Task | Signals | Decision |
|---|---:|---|
| FAQ | 0 | keep plain Python |
| code refactor | 2 | evaluate a graph |
| research brief | 6 | evaluate a graph |
| incident postmortem | 6 | evaluate a graph |

The FAQ has no durable transition to model. The research brief and incident postmortem do: both must preserve intermediate artifacts, explain recovery, and support review. The refactor sits on the boundary and should start with a plain staged implementation.

## Baseline before the framework

The fixture-backed comparison runs the same AtlasVector question in three forms. The staged pipeline is still ordinary Python; its event record reveals the state that Chapter 02 will turn into graph channels.


In [2]:
#| tbl-cap: "**Framework boundary.** The same deterministic task under three control models."
from evidence_brief.baselines import compare_baselines

comparison = compare_baselines("conflict-01")
rows = "\n".join(
    f"| {r['approach']} | {r['complete']} | {r['unsupported']} | {r['resume_keys']} | {r['events']} |"
    for r in comparison
)
display(Markdown(
    "| Approach | Complete | Unsupported claims | Resume keys | Events |\n"
    "|---|:---:|---:|---:|---:|\n" + rows
))
assert comparison[0]["unsupported"] > comparison[-1]["unsupported"] == 0
assert comparison[-1]["complete"] is True


| Approach | Complete | Unsupported claims | Resume keys | Events |
|---|:---:|---:|---:|---:|
| one shot | True | 2 | 0 | 1 |
| skill loop | True | 1 | 1 | 4 |
| staged pipeline | True | 0 | 5 | 9 |

The graph is not justified by answer quality alone. It is justified by the combination of zero unsupported claims, explicit resume state, and an inspectable trajectory. Chapter 02 keeps that contract and replaces the informal event sequence with legal graph transitions.
